In [5]:
import numpy as np
import scipy.linalg
import scipy.signal as signal
import soundfile as sf
import matplotlib.pyplot as plt
import os
import sys
import time

# Zorg ervoor dat de paden kloppen met je lokale mappenstructuur
current_dir = os.path.dirname(os.path.abspath(__file__))
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

from package import load_rirs
from package.utils import create_micsigs as original_create_micsigs

NameError: name '__file__' is not defined

Om de GSC goed te laten werken in een tijdvariërend scenario, moeten we continu de Direction of Arrival (DOA) schatten over de tijd. Hier gebruiken we een wideband MUSIC-algoritme voor. Deze functie helpt de analyse-stap te vervullen voor zowel DOA-schatting als de latere GSC-processing.

In [ ]:
def music_wideband(micsigs, fs, acoustic_scenario):
    L = 1024
    overlap = L // 1.5
    
    stft_list = []
    for m in range(micsigs.shape[1]):
        freqs, _, Zxx = signal.stft(micsigs[:, m], fs=fs, window='hann', nperseg=L, noverlap=overlap)
        stft_list.append(Zxx)
    stft_data = np.stack(stft_list, axis=0) # Shape: (M, nF, nT)
    
    M, nF, nT = stft_data.shape
    c = 343.0
    num_audio = acoustic_scenario.audioPos.shape[0] if acoustic_scenario.audioPos is not None else 0
    num_noise = acoustic_scenario.noisePos.shape[0] if acoustic_scenario.noisePos is not None else 0
    Q = num_audio + num_noise
    
    angles = np.arange(0, 180.5, 0.5)
    rads = np.radians(angles)
    
    mics_centered = acoustic_scenario.micPos - np.mean(acoustic_scenario.micPos, axis=0)
    px = mics_centered[:, 0].reshape(-1, 1)
    py = mics_centered[:, 1].reshape(-1, 1)
    
    valid_indices = range(1+L//15, L // 3)
    pseudospectra = []
    
    for k in valid_indices:
        Y = stft_data[:, k, :]
        Ryy = (Y @ Y.conj().T) / nT
        _, eigvecs = np.linalg.eigh(Ryy)
        En = eigvecs[:, :M-Q] 
        
        omega = 2 * np.pi * freqs[k]
        taus = (px * np.sin(rads) + py * np.cos(rads)) / c 
        A = np.exp(-1j * omega * taus)
        
        denom = np.sum(np.abs(En.conj().T @ A)**2, axis=0)
        p_theta = 1.0 / denom
        pseudospectra.append(p_theta)
        
    pseudospectra = np.array(pseudospectra) 
    
    log_p = np.log(pseudospectra)
    p_geom = np.exp(np.mean(log_p, axis=0))
    spectrum_geom_db = 10 * np.log10(p_geom / np.max(p_geom))
    
    peaks_indices, _ = signal.find_peaks(spectrum_geom_db)
    if len(peaks_indices) >= Q:
        sorted_peak_indices = peaks_indices[np.argsort(spectrum_geom_db[peaks_indices])][-Q:]
        estimated_doas = np.sort(angles[sorted_peak_indices])
    else:
        sorted_all = np.argsort(spectrum_geom_db)[::-1]
        est_idx = []
        for idx in sorted_all:
            if len(est_idx) == Q: break
            if all(abs(idx - e) > 20 for e in est_idx):
                est_idx.append(idx)
        estimated_doas = np.sort(angles[est_idx])

    ps_ind_db = 10 * np.log10(pseudospectra / np.max(pseudospectra, axis=1, keepdims=True))

    return angles, ps_ind_db, spectrum_geom_db, estimated_doas, stft_data


> Voor de GSC in individuele frequentiebins vervangen we de standaard DAS beamformer door een 'Filter-and-Sum Beamformer' (FAS BF). We maken vooraf een Look-Up Table aan met stuurvectoren door de DFT van de RIRs te normaliseren ten opzichte van de eerste microfoon:
> 
> 
> $$h(\omega,\theta)=\frac{a(\omega,\theta)}{a_{1}(\omega,\theta)}$$
> 
> 
> De FAS BF definiëren we vervolgens als:
> 
> 
> $$w_{FAS}(\omega,\theta)=\frac{h(\omega,\theta)}{h^{H}(\omega,\theta)h(\omega,\theta)}$$
> 
> 
> En de Blocking Matrix wordt zodanig gedefinieerd dat:
> 
> 
> $$B(\omega,\theta)h(\omega,\theta)=0$$
> 


In [ ]:
def get_target_rir_from_scenario(scenario, source_index=0):
    if not hasattr(scenario, 'RIRs_audio') or scenario.RIRs_audio is None:
        raise ValueError("Kan 'RIRs_audio' niet vinden in het scenario!")
    target_rir = scenario.RIRs_audio[:, :, source_index]
    return target_rir

def build_lut_for_target(target_rir, L=1024):
    n_bins = L // 2 + 1
    M_mics = target_rir.shape[1]
    
    H_omega = np.fft.rfft(target_rir, n=L, axis=0)
    
    W_FAS = np.zeros((n_bins, M_mics), dtype=complex)
    B_matrix = np.zeros((n_bins, M_mics - 1, M_mics), dtype=complex)
    
    for k in range(n_bins):
        h_k = H_omega[k, :].reshape(M_mics, 1)
        
        eps = 1e-12
        A_1 = h_k[0, 0]
        if np.abs(A_1) > eps:
            h_k = h_k / A_1
            
        denom = (h_k.conj().T @ h_k)[0, 0]
        if np.abs(denom) > eps:
            W_FAS[k, :] = (h_k / denom).flatten()
        else:
            W_FAS[k, :] = np.ones(M_mics) / M_mics
            
        Z = scipy.linalg.null_space(h_k.conj().T)
        if Z.shape[1] > 0:
            B_matrix[k, :, :] = Z.conj().T
            
    return W_FAS, B_matrix

def build_lut_all_angles(all_rirs_dict, angles, L=1024):
    lut = {}
    for angle in angles:
        rir = all_rirs_dict[angle]
        W_FAS, B = build_lut_for_target(rir, L=L)
        lut[angle] = (W_FAS, B)
    return lut

Deze implementatie transformeert tijddomein signalen met een STFT naar het frequentiedomein (met 50% overlap), voert in elke bin de GSC uit met de LUT, en zet de output via inverse DFT (WOLA) weer terug naar het tijdsdomein

In [ ]:
def gsc_fd(scenario, estimated_doa, lut, precomputed_speech, precomputed_noise, mu=0.1):
    speech = precomputed_speech
    noise = precomputed_noise
    mic = speech + noise

    lut_angles = np.array(list(lut.keys()))
    closest_angle = lut_angles[np.argmin(np.abs(lut_angles - estimated_doa))]
    W_FAS_lut, B_lut = lut[closest_angle]
    
    snr_in = 10 * np.log10(np.var(speech[:, 0]) / np.var(noise[:, 0]))
    vad = np.abs(speech[:, 0]) > (np.std(speech[:, 0]) * 0.1)
    
    N_samples, M_mics = mic.shape
    L = 1024
    overlap = L // 2
    window_sqrthann = np.sqrt(signal.windows.hann(L, sym=False))
  
    _, _, Zxx_mix = signal.stft(mic.T, fs=scenario.fs, window=window_sqrthann, nperseg=L, noverlap=overlap)
    _, _, Zxx_tar = signal.stft(speech.T, fs=scenario.fs, window=window_sqrthann, nperseg=L, noverlap=overlap)
    _, _, Zxx_int = signal.stft(noise.T, fs=scenario.fs, window=window_sqrthann, nperseg=L, noverlap=overlap)
    
    nF, nT = Zxx_mix.shape[1], Zxx_mix.shape[2]

    vad_frames = np.zeros(nT)
    samples_per_frame = L - overlap
    for n in range(nT):
        start_idx = n * samples_per_frame
        end_idx = start_idx + L
        if end_idx <= N_samples:
            vad_frames[n] = 1 if np.mean(vad[start_idx:end_idx]) > 0.5 else 0

    E_out_mix = np.zeros((nF, nT), dtype=complex)
    E_out_tar = np.zeros((nF, nT), dtype=complex) 
    E_out_int = np.zeros((nF, nT), dtype=complex) 
    FAS_out   = np.zeros((nF, nT), dtype=complex) 
    
    eps = 1e-8
    for k in range(nF):
        w_fas = W_FAS_lut[k, :]
        B_k = B_lut[k, :, :]
        w_nlms = np.zeros((M_mics - 1,), dtype=complex)
        
        for n in range(nT):
            x_mix = Zxx_mix[:, k, n]
            x_tar = Zxx_tar[:, k, n]
            x_int = Zxx_int[:, k, n]
            
            y_fas_mix = np.vdot(w_fas, x_mix)
            u_mix = B_k @ x_mix
            e_mix = y_fas_mix - np.vdot(w_nlms, u_mix)
            E_out_mix[k, n] = e_mix
            FAS_out[k, n] = y_fas_mix
            
            y_fas_tar = np.vdot(w_fas, x_tar)
            u_tar = B_k @ x_tar
            E_out_tar[k, n] = y_fas_tar - np.vdot(w_nlms, u_tar) 

            y_fas_int = np.vdot(w_fas, x_int)
            u_int = B_k @ x_int
            E_out_int[k, n] = y_fas_int - np.vdot(w_nlms, u_int) 

            if vad_frames[n] == 0:
                power = np.vdot(u_mix, u_mix).real
                w_nlms += mu * u_mix * np.conj(e_mix) / (power + eps)

    _, gsc_out_mix = signal.istft(E_out_mix, fs=scenario.fs, window=window_sqrthann, nperseg=L, noverlap=overlap)
    _, fas_out_time = signal.istft(FAS_out, fs=scenario.fs, window=window_sqrthann, nperseg=L, noverlap=overlap)
    _, out_tar = signal.istft(E_out_tar, fs=scenario.fs, window=window_sqrthann, nperseg=L, noverlap=overlap)
    _, out_int = signal.istft(E_out_int, fs=scenario.fs, window=window_sqrthann, nperseg=L, noverlap=overlap)
    
    return gsc_out_mix[:N_samples], mic, snr_in, fas_out_time[:N_samples], out_tar[:N_samples], out_int[:N_samples], speech, noise

Om de bereikte performantie aan te tonen en te kwantificeren, maken we gebruik van de Signal-to-Interference Ratio (SIR). De SIR kwantificeert de bijdrage van het gewenste signaal ten opzichte van het interferentiesignaal

In [ ]:
def compute_sir(y, x1, x2, groundTruth):
    if np.sqrt(np.sum((y - x1 - x2) ** 2)) / np.sqrt(np.sum(y ** 2)) > 0.01:
        sir = np.nan
    elif np.sum(groundTruth) + np.sum(1 - groundTruth) != len(groundTruth):
        sir = np.nan
    else:
        sir = 10 * np.log10(
            np.var(x1 * groundTruth + x2 * (1 - groundTruth)) /\
                np.var(x2 * groundTruth + x1 * (1 - groundTruth))
        )
    return sir

In dit experiment testen we een tijdsafhankelijk akoestisch scenario. We kiezen 5 posities voor de bron (180∘ tot 90∘) en 5 voor ruis (90∘ tot 0∘), met posities die elke 10 seconden wijzigen. We evalueren dit onder zowel anechoïsche (T60=0s) als reverberante (T60=1s) omstandigheden. Uiteindelijk plotten we de SIR waarden in de microfoonsignalen vergeleken met ná de GSC filtering

In [ ]:
# Zorg dat deze paden verwijzen naar jouw .wav en .pkl bestanden!
fs = 44100
segment_duration = 10.0 
segment_samples = int(fs * segment_duration)

t_dry, _ = sf.read(os.path.join(parent_dir, "sound_files", "part1_track1_dry.wav"))
i_dry, _ = sf.read(os.path.join(parent_dir, "sound_files", "part1_track2_dry.wav"))

t_dry = t_dry / np.max(np.abs(t_dry)) * 0.9
i_dry = i_dry / np.max(np.abs(i_dry)) * 0.9

for use_reverb in [False, True]:
    reverb_label = "REVERBERANT (T60=1s)" if use_reverb else "ANECHOIC (T60=0s)"
    
    if not use_reverb:
        rir_files = ["160_20_no_reverb.pkl", "140_40_no_reverb.pkl", "120_60_no_reverb.pkl", "100_40_no_reverb.pkl", "95_85_no_reverb.pkl"]
    else:
        rir_files = ["160_20_reverb.pkl", "140_40_reverb.pkl", "120_60_reverb.pkl", "100_40_reverb.pkl", "95_85_reverb.pkl"]

    all_rirs_dict = {}
    lut_angles_list = []
    
    # Let op: De RIRs in je mappen moeten qua naam overeenkomen!
    for rir_file in rir_files:
        scenario_tmp = load_rirs(os.path.join(parent_dir, "rirs", rir_file))
        angle_deg = float(rir_file.split("_")[0])
        all_rirs_dict[angle_deg] = scenario_tmp.RIRs_audio[:, :, 0]
        lut_angles_list.append(angle_deg)

    lut = build_lut_all_angles(all_rirs_dict, lut_angles_list, L=1024)

    full_mix_out = []
    full_mic_in = [] 
    sir_in_per_sec = []
    sir_out_per_sec = []
    
    print("\n" + "="*70)
    print(f"STARTING SCENARIO: {reverb_label}")
    print("="*70)
    
    for i, rir_file in enumerate(rir_files):
        print(f"\n--- Segment {i+1}/5 | Tijd: {i*10}s tot {(i+1)*10}s | Bestand: {rir_file} ---")
        
        scenario = load_rirs(os.path.join(parent_dir, "rirs", rir_file))

        target_rir = scenario.RIRs_audio[:, :, 0]
        interf_rir = scenario.RIRs_audio[:, :, 1] if scenario.RIRs_audio.shape[2] > 1 else scenario.RIRs_noise[:, :, 0]

        chunk_t = t_dry[i*segment_samples : (i+1)*segment_samples]
        chunk_i = i_dry[i*segment_samples : (i+1)*segment_samples]

        t_comp = np.stack([signal.fftconvolve(chunk_t, target_rir[:,m], mode='full')[:segment_samples] for m in range(5)], axis=1)
        i_comp = np.stack([signal.fftconvolve(chunk_i, interf_rir[:,m], mode='full')[:segment_samples] for m in range(5)], axis=1)
        mic_mix = t_comp + i_comp
        
        _, _, _, est_doas, _ = music_wideband(mic_mix, fs, scenario)
        target_doa_est = np.max(est_doas)
        print(f"   [DOA] MUSIC detecteert bronnen op: {est_doas}°")
        
        gsc_out_mix, mic, _, _, out_tar, out_int, _, _ = gsc_fd(
            scenario, estimated_doa=target_doa_est, lut=lut,                      
            precomputed_speech=t_comp, precomputed_noise=i_comp, mu=0.1
        )
        
        full_mix_out.append(gsc_out_mix)
        full_mic_in.append(mic[:, 0]) 
        
        sec_samples = int(fs * 1.0)
        ground_truth = np.ones(sec_samples, dtype=int) 
        
        for s in range(10): 
            start_idx = s * sec_samples
            end_idx = start_idx + sec_samples
            
            y_in_sec = t_comp[start_idx:end_idx, 0] + i_comp[start_idx:end_idx, 0]
            sir_in = compute_sir(y_in_sec, t_comp[start_idx:end_idx, 0], i_comp[start_idx:end_idx, 0], ground_truth)
            sir_out = compute_sir(gsc_out_mix[start_idx:end_idx], out_tar[start_idx:end_idx], out_int[start_idx:end_idx], ground_truth)
            
            sir_in_per_sec.append(sir_in)
            sir_out_per_sec.append(sir_out)

    # De visuele Output Genereren
    plt.figure(figsize=(12, 6))
    t_axis = np.arange(1, 51)
    plt.plot(t_axis, sir_in_per_sec, color='gray', linestyle=':', label='Input SIR (Mic 1)')
    plt.plot(t_axis, sir_out_per_sec, color='blue', linewidth=2, label='Output SIR (GSC)')
    
    for wissel in range(10, 50, 10):
        plt.axvline(x=wissel, color='red', linestyle='--', alpha=0.5)
        
    plt.title(f'SIR Verbetering over Tijd - {reverb_label}')
    plt.ylabel('SIR (dB)')
    plt.xlabel('Tijd (seconden)')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()